# ROOT TH2 maps in Square-Dalitz coordinates

For B decays, efficiency/background maps are often stored directly in $(m',\theta')$. This example uses the full paper-inspired $B^+\to K^+\pi^+\pi^-$ benchmark: $K^*(892)^0+(K\pi)_S+\rho(770)^0+f_0(980)+NR$. The ROOT helpers convert invariant coordinates internally, while `generate_toy` and `FitSession` use the maps exactly like ordinary efficiencies/backgrounds.


In [ ]:
import numpy as np
import uproot
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    BaBarFlatte, BackgroundSpec, DecayChannel, DecayModel, FitSession, LASS,
    NonResonant, Parameter, RealImag, Resonance, ToyBackground, enable_x64,
    generate_toy, plot_dalitz, plot_square_dalitz,
    square_dalitz_background_from_root, square_dalitz_efficiency_from_root,
)
enable_x64()


In [ ]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))
truth_xy = {
    "Kstar892": (1.00, 0.00),
    "KpiS": (1.40, -0.60),
    "rho770": (0.65, 0.10),
    "f0_980": (-0.20, 1.00),
    "NR": (-0.50, 0.10),
}
truth = {}

def coefficient(name, fixed=False):
    x, y = truth_xy[name]
    if fixed:
        return RealImag(x, y)
    truth[f"{name}.x"] = x
    truth[f"{name}.y"] = y
    return RealImag(
        Parameter.coefficient(f"{name}.x", x, owner=name, step=0.01),
        Parameter.coefficient(f"{name}.y", y, owner=name, step=0.01),
    )

c = {name: coefficient(name, fixed=(name == "Kstar892")) for name in truth_xy}
model = DecayModel(
    channel,
    [
        Resonance("Kstar892", (0,2), c["Kstar892"], mass=0.8958, width=0.0474, spin=1, resonance_radius=4.0, parent_radius=4.0),
        Resonance("KpiS", (0,2), c["KpiS"], lineshape=LASS(2.07, 3.32, 1.8), mass=1.425, width=0.270, spin=0, resonance_radius=4.0, parent_radius=4.0),
        Resonance("rho770", (1,2), c["rho770"], mass=0.7753, width=0.1491, spin=1, resonance_radius=4.0, parent_radius=4.0),
        Resonance("f0_980", (1,2), c["f0_980"], lineshape=BaBarFlatte(), mass=0.965, width=0.0, spin=0, resonance_radius=4.0, parent_radius=4.0),
        NonResonant(c["NR"]),
    ],
    normalization_method="square-dalitz",
    normalization_resolution=320,
    normalization_pair=(0,2),
)


In [ ]:
edges = np.linspace(0,1,26)
cbin = 0.5*(edges[:-1]+edges[1:])
MP,TP = np.meshgrid(cbin,cbin,indexing="ij")
eff_values = 0.45 + 0.35*MP + 0.10*np.cos(np.pi*TP)
bkg_values = 0.30 + 0.70*TP

with uproot.recreate("maps_sdp.root") as f:
    f["efficiency_sdp"] = (eff_values,edges,edges)
    f["background_sdp"] = (bkg_values,edges,edges)

kwargs = dict(mother_mass=model.channel.parent_mass, masses=model.channel.daughter_masses, pair=(0,2))
eff = square_dalitz_efficiency_from_root("maps_sdp.root","efficiency_sdp",**kwargs)
bkg = square_dalitz_background_from_root("maps_sdp.root","background_sdp",**kwargs)


In [ ]:
f_sig = Parameter("signal_fraction",0.76,bounds=(0.05,0.99))
data = generate_toy(
    model, 30_000, parameters=truth, efficiency=eff, signal_fraction=0.82,
    backgrounds=(ToyBackground("comb",bkg),), seed=1515,
)
plot_square_dalitz(data, **kwargs, title="Toy in Square Dalitz")
plt.show()
plot_dalitz(data,x="s13",y="s23",title="Same toy in ordinary Dalitz")
plt.show()

session = FitSession(model,data,efficiency=eff,signal_fraction=f_sig, backgrounds=(BackgroundSpec("comb",bkg),))
start = {p.name: truth[p.name] + 0.08 for p in session.parameters if p.name in truth}
start["signal_fraction"] = 0.74
result = session.fit(start, simplex=True, ncall=55_000)
session.report(result, acceptance_weighted_fractions=True)
session.plot_projection(result,"s13")
plt.show()
